In [ ]:
# conda activate parking
# pip install jupyter notebook pandas pillow google-generativeai python-dotenv requests tqdm ipywidgets

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import json
import re
import warnings
from io import BytesIO
from pathlib import Path
from typing import Optional
from tqdm.notebook import tqdm  # ← notebook용으로 변경
import google.generativeai as genai
from PIL import Image
import sys

In [ ]:
# ==========================
# 1. 설정
# ==========================

# .env 파일에서 API 키 로드
load_dotenv()
KAKAO_REST_API_KEY = os.getenv('KAKAO_API_KEY')

# API 키 유효성 검사
api_key_valid = KAKAO_REST_API_KEY and 'YOUR_API_KEY' not in KAKAO_REST_API_KEY
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 KAKAO_API_KEY를 실제 API 키로 설정해주세요!")
    raise ValueError("KAKAO_API_KEY를 실제 카카오 REST API 키로 변경해주세요.")

TXT_ROUND_PATH = "mountain_round.txt"   # mountain_round.txt 경로
TXT_ONEWAY_PATH = "mountain_oneway.txt"  
OUTPUT_ROUND_CSV = "result/등하산_주차장.csv"
OUTPUT_ONEWAY_CSV = "result/편도_주차장.csv"
SEARCH_RADIUS_M = 2000             # 2km
CATEGORY_CODE = "PK6"              # 주차장
MAX_PAGES = 3                      # 15 * 3 = 45개
PAGE_SIZE = 15

BASE_URL = "https://dapi.kakao.com/v2/local/search/category.json"

headers = {
    "Authorization": f"KakaoAK {KAKAO_REST_API_KEY}"
}

</br>

# 함수

In [ ]:
def parse_mountain_and_trail_code(filename: str):
    """
    예) '대암산_0000000001.gpx' -> ('대암산', '대암산_01')
        '지리산(통영)_0000000002.gpx' -> ('지리산(통영)', '지리산(통영)_02')
    """
    name_no_ext = os.path.splitext(filename)[0]  # '대암산_0000000001'
    parts = name_no_ext.split("_")
    serial_raw = parts[-1]                      # '0000000001'
    mountain_name = "_".join(parts[:-1])        # '대암산' 또는 '지리산(통영)'

    # 숫자로 안전하게 변환해서 2자리 코드로
    try:
        serial_num = int(serial_raw)
        serial_str = f"{serial_num:02d}"
    except ValueError:
        # 혹시 숫자가 아니면 앞의 0 제거 정도만
        serial_str = serial_raw.lstrip("0") or serial_raw

    trail_code = f"{mountain_name}_{serial_str}"
    return mountain_name, trail_code


def get_parkings_nearby(lon: float, lat: float, radius: int = SEARCH_RADIUS_M):
    """
    카카오 카테고리 검색 API로 중심좌표(lon, lat) 기준 radius m 내 주차장 리스트 가져오기.
    x = 경도(lng), y = 위도(lat)
    """
    all_docs = []

    for page in range(1, MAX_PAGES + 1):
        params = {
            "category_group_code": CATEGORY_CODE,
            "x": lon,
            "y": lat,
            "radius": radius,
            "sort": "distance",  # 거리순
            "page": page,
            "size": PAGE_SIZE
        }

        res = requests.get(BASE_URL, headers=headers, params=params, timeout=5)
        res.raise_for_status()
        data = res.json()

        documents = data.get("documents", [])
        if not documents:
            break

        all_docs.extend(documents)

        meta = data.get("meta", {})
        is_end = meta.get("is_end", True)
        if is_end:
            break

        time.sleep(0.2)

    return all_docs

# -------------------------------------

def iter_base_points(row, mode: str):
    """
    mode:
      - "oneway": start만
      - "round": start, end 둘 다
    반환: (base_type, lat, lon) generator
    """
    start_lat = float(row["출발_lat"])
    start_lon = float(row["출발_lon"])

    if mode == "oneway":
        yield "start", start_lat, start_lon
        return

    if mode == "round":
        end_lat = float(row["도착_lat"])
        end_lon = float(row["도착_lon"])
        yield "start", start_lat, start_lon
        yield "end", end_lat, end_lon
        return

    raise ValueError(f"Unknown mode: {mode}")


def collect_parkings_for_file(df, mode: str, sleep_sec: float = 0.3):
    results = []

    for _, row in df.iterrows():
        filename = row["파일명"]
        mountain_name, trail_code = parse_mountain_and_trail_code(filename)

        print(f"\n=== 등산로 처리: {filename} ({mountain_name}, {trail_code}) ===")

        for base_type, lat, lon in iter_base_points(row, mode=mode):
            print(f" ▶ [{base_type}] 기준 좌표({lat}, {lon}) 2km 내 주차장 검색 중...")
            docs = get_parkings_nearby(lon=lon, lat=lat, radius=SEARCH_RADIUS_M)
            print(f"    → 수집된 주차장 수 ({base_type}): {len(docs)}개")

            for d in docs:
                address = d.get("road_address_name") or d.get("address_name")
                distance_m = int(d.get("distance", 0)) if d.get("distance") is not None else None

                results.append({
                    "mountain_name": mountain_name,
                    "trail_code": trail_code,
                    "base_type": base_type,          
                    "place_id": d.get("id"),
                    "place_name": d.get("place_name"),
                    "category": "주차장",
                    "tour_spot_type": np.nan,
                    "lat": float(d.get("y")),
                    "lng": float(d.get("x")),
                    "distance_m": distance_m,
                    "address": address,
                    "tel": d.get("phone"),
                    "url": d.get("place_url"),
                })

            time.sleep(sleep_sec)

    return results


def save_results(results, output_csv_path: str):
    if not results:
        print("\n=== 수집된 주차장 데이터가 없습니다. ===")
        return

    col_order = [
        "mountain_name", "trail_code", "base_type",
        "place_id", "place_name", "category", "tour_spot_type",
        "lat", "lng", "distance_m", "address", "tel", "url"
    ]

    df_result = pd.DataFrame(results)

    # 혹시 누락 컬럼이 생겨도 안전하게 처리
    for c in col_order:
        if c not in df_result.columns:
            df_result[c] = np.nan

    df_result = df_result[col_order]

    print(f"\n=== 최종 수집된 주차장 레코드 수: {len(df_result)} ===")
    df_result.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    print(f"→ '{output_csv_path}' 파일로 저장 완료")


def main(mode: str):
    if mode == "oneway":
        df = pd.read_csv(TXT_ONEWAY_PATH, sep="\t")
        output_path = OUTPUT_ONEWAY_CSV
    elif mode == "round":
        df = pd.read_csv(TXT_ROUND_PATH, sep="\t")
        output_path = OUTPUT_ROUND_CSV
    else:
        raise ValueError("mode must be 'oneway' or 'round'")

    results = collect_parkings_for_file(df, mode=mode, sleep_sec=0.3)
    save_results(results, output_path)

## 1. 등하산(왕복) 등산로 반경 2km 주차장 poi 수집

---

**주요 기능:**

- 카카오 로컬 API를 사용해 각 지점(출발지/도착지) 2km 반경 내 주차장 검색

- 주차장명, 좌표, 거리, 주소, 연락처 등 상세 정보 수집

In [ ]:
"""
등산로 주차장 정보 수집 스크립트

[주요 기능]
- mountain_round.txt 파일에서 등산로별 출발/도착 좌표 읽기
- 카카오 로컬 API를 사용해 각 지점 2km 반경 내 주차장 검색
- 주차장명, 좌표, 거리, 주소, 연락처 등 상세 정보 수집
- 결과를 CSV 파일로 저장 (등하산_주차장.csv)

[입력 파일 형식]
- mountain_round.txt: 탭으로 구분된 파일
  컬럼: 파일명, 출발_lat, 출발_lon, 도착_lat, 도착_lon

[출력]
- result/등하산_주차장.csv
  각 등산로의 출발(start)/도착(end) 지점별 주차장 정보
"""

if __name__ == "__main__":
    main(
        mode = 'round'
        )

</br></br>

## 2. 편도 등산로 반경 2km 주차장 poi 수집

---

In [ ]:
"""
편도 등산로 주차장 정보 수집 스크립트

[주요 기능]
- mountain_oneway.txt 파일에서 등산로별 출발 좌표 읽기
- 카카오 로컬 API를 사용해 출발 지점 2km 반경 내 주차장 검색
- 주차장명, 좌표, 거리, 주소, 연락처 등 상세 정보 수집
- 결과를 CSV 파일로 저장 (편도_주차장.csv)

[입력 파일 형식]
- mountain_oneway.txt: 탭으로 구분된 파일
  컬럼: 파일명, 출발_lat, 출발_lon, 도착_lat, 도착_lon
  (편도 등산로는 출발 좌표만 사용)

[출력]
- result/편도_주차장.csv
  각 등산로의 출발 지점 주변 주차장 정보
"""

if __name__ == "__main__":
    main(
        mode = 'oneway'
    )

</br>

# 주차장 데이터 병합

In [ ]:
df_round_parking = pd.read_csv("등하산_주차장.csv")
df_oneway_parking = pd.read_csv("편도_주차장.csv")

# 왕복코스(등하산_원점회귀, 등하산_횡단) + 편도코스(편도_등산, 편도_하산) concat 
df_parking = pd.concat([df_round_parking, df_oneway_parking], ignore_index=True)
df_parking = df_parking[df_parking['base_type'] != "end"].copy()
df_parking.to_csv("result/등산로_주차장.csv")
df_parking.head(3)

,mountain_name,trail_code,base_type,place_id,place_name,category,tour_spot_type,lat,lng,distance_m,address,tel,url
0,가리산,가리산_02,start,22756797,가리산자연휴양림 주차장,주차장,NaN,37.864412,127.984483,25,강원특별자치도 홍천군 두촌면 가리산길 426,NaN,http://place.map.kakao.com/22756797
1,가리산,가리산_02,start,768314002,가리산자연휴양림 제2주차장,주차장,NaN,37.865413,127.981470,263,강원특별자치도 홍천군 두촌면 가리산길 426,NaN,http://place.map.kakao.com/768314002
6,가야산,가야산_03,start,20848683,가야산국립공원 백운동주차장,주차장,NaN,35.800984,128.141777,63,경북 성주군 수륜면 백운리 1805,NaN,http://place.map.kakao.com/20848683


</br></br>
 
## 주차장 사용가능여부 자동검증

---

**주요 기능:**

- 텍스트 사전 필터링: 키워드 기반으로 즉시 판단 (공영주차장, 식당 등)

- Gemini AI 분석: 위성 이미지를 분석하여 사용 가능 여부 판단

- 비용 최적화: 텍스트 필터링으로 API 호출 절감

In [ ]:
"""
등산로 주차장 검증 스크립트
- 좌표 기반 지도 이미지를 가져와 Gemini Vision으로 분석
- 주변 건물 유형을 파악하여 실제 사용 가능한 주차장인지 판별

주차장 사용 가능 여부 AI 검증 스크립트

[주요 기능]
1. 텍스트 사전 필터링: 키워드 기반으로 즉시 판단 (공영주차장, 식당 등)
2. Gemini AI 분석: 위성 이미지를 분석하여 사용 가능 여부 판단
3. 비용 최적화: 텍스트 필터링으로 약 60%의 API 호출 절감

[분석 방법]
- 1단계: 텍스트 키워드 필터링 
  * 확실히 사용 가능: 공영주차장, 국립공원, 자연휴양림 등
  * 확실히 사용 불가: 식당, 카페, 교회, 병원, 아파트 등
  
- 2단계: Gemini AI 분석 (이미지 또는 텍스트)
  * 이미지 분석: Google/Kakao 지도 API로 위성사진 수집 후 AI 판단
  * 텍스트 분석: 이미지 없이 장소명과 주소만으로 판단
"""

# Python 버전 경고 억제 (Python 3.8 사용 시)
warnings.filterwarnings('ignore', category=FutureWarning, module='google.api_core')

# Python 버전 확인
import sys
PYTHON_VERSION = sys.version_info
if PYTHON_VERSION < (3, 8):
    print("⚠️ 경고: Python 3.8 이상이 필요합니다.")
    sys.exit(1)

# .env 파일에서 환경변수 로드
load_dotenv()

# API 키 설정 (공백 제거)
def get_api_key(key_name: str) -> Optional[str]:
    """API 키를 가져오고 공백을 제거"""
    key = os.getenv(key_name)
    if key:
        key = key.strip()  # 앞뒤 공백 제거
        # 빈 문자열이면 None 반환
        if not key:
            return None
    return key

GOOGLE_MAPS_API_KEY = get_api_key("GOOGLE_MAPS_API_KEY")  # 지도 이미지용
KAKAO_API_KEY = get_api_key("KAKAO_API_KEY")              # 지도 이미지 대체
GEMINI_API_KEY = get_api_key("GEMINI_API_KEY")            # 🔥 Gemini API


# ============================================================
# 💰 비용 절감 핵심: 텍스트 기반 사전 필터링
# ============================================================

# 확실히 사용 가능한 키워드 (이미지 분석 불필요)
DEFINITELY_AVAILABLE_KEYWORDS = [
    "공영주차장", "공공주차장", "공용주차장",
    "국립공원", "도립공원", "군립공원", "시립공원",
    "자연휴양림", "휴양림",
    "등산로", "등산객",
    "관광지", "관광안내소",
    "출렁다리", "케이블카",
    "매표소", "탐방로", "탐방안내소",
    "야영장", "캠핑장",
    "산림청", "국유림",
]

# 확실히 사용 불가능한 키워드 (이미지 분석 불필요)
DEFINITELY_UNAVAILABLE_KEYWORDS = [
    "식당", "음식점", "레스토랑", "맛집",
    "카페", "커피",
    "교회", "성당", "절", "사찰", "암자",
    "아파트", "빌라", "오피스텔", "주상복합",
    "회사", "본사", "지사", "사옥",
    "병원", "의원", "클리닉", "치과", "한의원",
    "학교", "대학교", "초등학교", "중학교", "고등학교",
    "마트", "슈퍼", "편의점",
    "호텔", "모텔", "펜션",  # 고객 전용일 가능성 높음
]


def prefilter_by_text(place_name: str, address: str = "") -> Optional[dict]:
    """
    텍스트 기반 사전 필터링 (이미지 분석 없이 판단)
    
    Returns:
        dict: 판단 결과 (확정된 경우)
        None: 이미지 분석 필요 (애매한 경우)
    """
    text = f"{place_name} {address}".lower()
    
    # 1. 확실히 사용 가능한 경우
    for keyword in DEFINITELY_AVAILABLE_KEYWORDS:
        if keyword in text:
            return {
                "is_available": True,
                "confidence": "high",
                "detected_buildings": [],
                "reason": f"텍스트 필터링: '{keyword}' 키워드 포함 → 공공시설로 판단",
                "filter_method": "text_prefilter"
            }
    
    # 2. 확실히 사용 불가능한 경우
    for keyword in DEFINITELY_UNAVAILABLE_KEYWORDS:
        if keyword in text:
            return {
                "is_available": False,
                "confidence": "high",
                "detected_buildings": [keyword],
                "reason": f"텍스트 필터링: '{keyword}' 키워드 포함 → 전용 주차장으로 판단",
                "filter_method": "text_prefilter"
            }
    
    # 3. 애매한 경우 → 이미지 분석 필요
    return None


# ============================================================
# 🔥 Gemini API (이미지 분석)
# ============================================================

def extract_json_from_response(text: str) -> dict:
    """
    Gemini 응답에서 JSON을 안전하게 추출
    다양한 형식의 응답을 처리할 수 있도록 여러 방법 시도
    """
    if not text:
        raise json.JSONDecodeError("Empty response", text or "", 0)
    
    original_text = text
    
    # 방법 1: ```json ... ``` 블록 추출
    json_match = re.search(r'```json\s*([\s\S]*?)\s*```', text)
    if json_match:
        text = json_match.group(1)
    # 방법 2: ``` ... ``` 블록 추출
    elif '```' in text:
        parts = text.split('```')
        if len(parts) >= 2:
            text = parts[1].strip()
            # json 태그 제거
            if text.startswith('json'):
                text = text[4:].strip()
    
    # 방법 3: { } 블록 직접 찾기
    if not text.strip().startswith('{'):
        brace_match = re.search(r'\{[\s\S]*\}', text)
        if brace_match:
            text = brace_match.group(0)
    
    text = text.strip()
    
    # JSON 파싱 시도
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # 방법 4: 불완전한 JSON 수정 시도
    # 끝에 } 또는 ]} 누락된 경우
    try:
        # 따옴표가 닫히지 않은 경우 처리
        if text.count('"') % 2 != 0:
            # 마지막 따옴표 위치 찾기
            last_quote = text.rfind('"')
            if last_quote > 0:
                # 마지막 값 뒤에 따옴표 추가
                text = text + '"'
        
        # 괄호 균형 맞추기
        open_braces = text.count('{')
        close_braces = text.count('}')
        open_brackets = text.count('[')
        close_brackets = text.count(']')
        
        if open_braces > close_braces:
            text = text + '}' * (open_braces - close_braces)
        if open_brackets > close_brackets:
            text = text.rstrip('}')  # 잠시 } 제거
            text = text + ']' * (open_brackets - close_brackets)
            text = text + '}' * (open_braces - close_braces)  # } 다시 추가
        
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # 방법 5: 필수 필드만 추출 시도
    try:
        is_available = None
        confidence = "low"
        reason = "JSON 파싱 실패로 원본 텍스트에서 추출"
        
        # is_available 추출
        if '"is_available"' in original_text.lower() or "'is_available'" in original_text.lower():
            if 'true' in original_text.lower():
                is_available = True
            elif 'false' in original_text.lower():
                is_available = False
        
        # reason 추출 시도
        reason_match = re.search(r'"reason"\s*:\s*"([^"]*)"', original_text)
        if reason_match:
            reason = reason_match.group(1)
        
        if is_available is not None:
            return {
                "is_available": is_available,
                "confidence": confidence,
                "detected_buildings": [],
                "reason": reason
            }
    except:
        pass
    
    # 모든 방법 실패
    raise json.JSONDecodeError(
        f"JSON 추출 실패. 원본 응답: {original_text[:200]}...",
        original_text,
        0
    )


def get_available_gemini_models():
    """
    API에서 실제로 사용 가능한 Gemini 모델 목록 조회
    """
    if not GEMINI_API_KEY:
        return []
    
    try:
        import google.generativeai as genai
        genai.configure(api_key=GEMINI_API_KEY)
        models = genai.list_models()
        available = []
        for m in models:
            if 'generateContent' in m.supported_generation_methods:
                # 'models/' 접두사 제거
                model_name = m.name.replace('models/', '')
                available.append(model_name)
        return available
    except:
        return []


def analyze_parking_with_gemini(
    image_bytes: bytes,
    place_name: str,
    address: str,
    model: str = None  # None이면 자동으로 사용 가능한 모델 선택
) -> dict:
    """
    🔥 Gemini API를 사용한 초저비용 이미지 분석
    
    Args:
        image_bytes: 이미지 바이트 데이터
        place_name: 장소명
        address: 주소
        model: 사용할 모델 (None이면 자동 선택)
    """
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY가 설정되지 않았습니다.")
    
    # API 키 형식 확인
    if len(GEMINI_API_KEY) < 20:
        raise ValueError(f"GEMINI_API_KEY가 너무 짧습니다. 올바른 API 키인지 확인하세요. (길이: {len(GEMINI_API_KEY)})")
    
    try:
        import google.generativeai as genai
        from PIL import Image
    except ImportError as e:
        error_msg = (
            f"google-generativeai 패키지가 설치되지 않았습니다.\n"
            f"다음 명령어로 설치하세요: pip install google-generativeai Pillow"
        )
        raise ImportError(error_msg) from e
    
    # GenerativeModel 실제 사용 가능 여부 확인
    try:
        # 실제로 GenerativeModel을 사용할 수 있는지 테스트
        test_model = genai.GenerativeModel
    except AttributeError:
        # 패키지 버전 확인
        try:
            import pkg_resources
            version = pkg_resources.get_distribution("google-generativeai").version
            error_msg = (
                f"google-generativeai 패키지 버전 ({version})이 Python 3.8과 호환되지 않을 수 있습니다.\n"
                f"다음 명령어로 최신 버전을 설치하세요:\n"
                f"  pip install --upgrade google-generativeai\n"
                f"또는 Python 3.10 이상을 사용하세요."
            )
        except:
            error_msg = (
                f"google-generativeai 패키지가 올바르게 설치되지 않았습니다.\n"
                f"다음 명령어로 재설치하세요:\n"
                f"  pip uninstall google-generativeai\n"
                f"  pip install google-generativeai"
            )
        raise AttributeError(error_msg)
    
    # API 키 설정 (재시도 로직 포함)
    try:
        genai.configure(api_key=GEMINI_API_KEY)
    except Exception as e:
        error_msg = (
            f"Gemini API 키 설정 실패: {str(e)}\n"
            f"API 키 형식 확인:\n"
            f"  - 길이: {len(GEMINI_API_KEY)} 문자\n"
            f"  - 시작 문자: {GEMINI_API_KEY[:5]}...\n"
            f"  - .env 파일에 GEMINI_API_KEY가 올바르게 설정되어 있는지 확인하세요."
        )
        raise ValueError(error_msg) from e
    
    prompt = f"""당신은 지도 이미지를 분석하여 주차장이 일반 대중이 실제로 사용할 수 있는지 판별하는 전문가입니다.

다음 주차장이 등산객이 실제로 사용할 수 있는 주차장인지 분석해주세요.

장소명: {place_name}
주소: {address}

판단 기준:
- 사용 불가: 식당, 카페, 교회, 사기업, 아파트, 병원 전용 주차장
- 사용 가능: 공영주차장, 국립공원, 자연휴양림, 관광지 주차장

이미지를 분석하여 다음 JSON 형식으로만 응답하세요:
{{"is_available": true/false, "confidence": "high/medium/low", "detected_buildings": ["건물유형"], "reason": "판단이유"}}"""

    try:
        # 기본 모델 목록 (우선순위 순)
        # 참고: Google AI Studio에서 무료로 사용 가능한 멀티모달 모델
        default_models = [
            "gemini-2.5-flash",     # ← 최신 모델 먼저
            "gemini-1.5-flash",
            "gemini-1.5-pro",
        ]
        
        # 실제 사용 가능한 모델 확인
        try:
            available_api_models = get_available_gemini_models()
            if available_api_models:
                # 사용 가능한 모델만 필터링
                available_models = [m for m in default_models if m in available_api_models]
                if not available_models:
                    # 기본 모델이 없으면 사용 가능한 첫 번째 모델 사용
                    available_models = available_api_models[:3]
            else:
                # 모델 목록 조회 실패 시 기본 모델 사용
                available_models = default_models
        except:
            # 오류 발생 시 기본 모델 사용
            available_models = default_models
        
        # 모델이 지정되지 않았으면 자동 선택
        if model is None:
            model = available_models[0] if available_models else default_models[0]
        
        # 이미지 로드
        image = Image.open(BytesIO(image_bytes))
        
        # 여러 모델 시도
        last_error = None
        max_retries = 2  # JSON 파싱 실패 시 재시도 횟수
        
        for try_model in available_models:
            if model and try_model != model:
                continue  # 지정된 모델만 시도
            
            for retry in range(max_retries + 1):
                try:
                    # Gemini 모델 호출
                    model_instance = genai.GenerativeModel(try_model)
                    response = model_instance.generate_content(
                        [prompt, image],
                        generation_config=genai.GenerationConfig(
                            temperature=0.1 + (retry * 0.1),  # 재시도 시 temperature 약간 증가
                            max_output_tokens=400,  # 충분한 토큰 확보
                        )
                    )
                    
                    result_text = response.text
                    
                    # 향상된 JSON 추출 함수 사용
                    result = extract_json_from_response(result_text)
                    result["filter_method"] = f"gemini_{try_model}"
                    return result
                    
                except json.JSONDecodeError as e:
                    # JSON 파싱 오류 시 재시도
                    if retry < max_retries:
                        time.sleep(0.5)  # 잠시 대기 후 재시도
                        continue
                    last_error = e
                    # 마지막 재시도도 실패하면 다음 모델 시도
                    break
                    
                except Exception as e:
                    last_error = e
                    error_str = str(e)
                    # NotFound 오류면 다음 모델 시도
                    if "NotFound" in error_str or "404" in error_str or "not found" in error_str.lower():
                        print(f"   [시도] {try_model} 실패, 다음 모델 시도...")
                        break  # 다음 모델로
                    # API 키 오류면 즉시 중단
                    if "API key" in error_str and ("invalid" in error_str.lower() or "not valid" in error_str.lower()):
                        raise
                    # 다른 오류면 즉시 중단
                    raise
        
        # 모든 모델 실패
        if last_error:
            raise last_error
            
    except ImportError as e:
        error_msg = str(e)
        print(f"\n❌ Gemini 패키지 오류: {error_msg}")
        return {
            "is_available": None,
            "confidence": "error",
            "detected_buildings": [],
            "reason": f"패키지 설치 필요: pip install google-generativeai Pillow",
            "filter_method": "gemini_error"
        }
    except AttributeError as e:
        error_msg = str(e)
        print(f"\n❌ Gemini 패키지 버전 오류: {error_msg}")
        return {
            "is_available": None,
            "confidence": "error",
            "detected_buildings": [],
            "reason": f"패키지 업그레이드 필요: pip install --upgrade google-generativeai",
            "filter_method": "gemini_error"
        }
    except json.JSONDecodeError as e:
        # JSON 파싱 오류 (Gemini 응답이 유효한 JSON이 아닌 경우)
        error_msg = str(e)
        print(f"\n⚠️ JSON 파싱 오류: {error_msg}")
        print(f"   → Gemini 응답이 유효한 JSON 형식이 아닙니다. 재시도하면 해결될 수 있습니다.")
        return {
            "is_available": None,
            "confidence": "error",
            "detected_buildings": [],
            "reason": f"JSON 파싱 오류 (재시도 권장): {error_msg}",
            "filter_method": "gemini_error"
        }
    except ValueError as e:
        # API 키 관련 오류
        error_msg = str(e)
        print(f"\n❌ API 키 오류: {error_msg}")
        print(f"\n💡 해결 방법:")
        print(f"   1. .env 파일 확인:")
        print(f"      - 파일 위치: {os.path.abspath('.env')}")
        print(f"      - GEMINI_API_KEY=your-actual-api-key 형식인지 확인")
        print(f"      - 따옴표 없이 입력했는지 확인")
        print(f"      - 앞뒤 공백이 없는지 확인")
        print(f"   2. 다른 터미널에서 테스트:")
        print(f"      python -c \"from dotenv import load_dotenv; import os; load_dotenv(); print('Key length:', len(os.getenv('GEMINI_API_KEY', '')))\"")
        return {
            "is_available": None,
            "confidence": "error",
            "detected_buildings": [],
            "reason": f"API 키 오류: {error_msg}",
            "filter_method": "gemini_error"
        }
    except Exception as e:
        error_type = type(e).__name__
        error_msg = str(e)
        
        # API 키 invalid 오류
        if "API key" in error_msg and ("invalid" in error_msg.lower() or "not valid" in error_msg.lower()):
            print(f"\n❌ Gemini API 키가 유효하지 않습니다: {error_msg}")
            print(f"\n💡 해결 방법:")
            print(f"   1. API 키 확인:")
            print(f"      - https://aistudio.google.com/apikey 에서 새 키 생성")
            print(f"      - .env 파일의 GEMINI_API_KEY 값 확인")
            print(f"      - 다른 곳에서 사용 중인 키와 동일한지 확인")
            print(f"   2. API 키 형식:")
            print(f"      - 현재 키 길이: {len(GEMINI_API_KEY) if GEMINI_API_KEY else 0} 문자")
            if GEMINI_API_KEY:
                print(f"      - 시작 부분: {GEMINI_API_KEY[:10]}...")
            print(f"   3. 환경변수 직접 확인:")
            print(f"      python -c \"import os; from dotenv import load_dotenv; load_dotenv(); key=os.getenv('GEMINI_API_KEY'); print('Key exists:', key is not None); print('Key length:', len(key) if key else 0)\"")
        # 모델을 찾을 수 없는 경우
        elif "NotFound" in error_msg or "404" in error_msg or "not found" in error_msg.lower():
            print(f"\n⚠️ Gemini 모델을 찾을 수 없습니다: {error_msg}")
            print(f"💡 해결 방법:")
            print(f"   1. API 키가 올바른지 확인하세요")
            print(f"   2. Gemini API가 활성화되어 있는지 확인하세요")
        else:
            print(f"\n⚠️ Gemini 분석 오류 ({error_type}): {error_msg}")
        
        return {
            "is_available": None,
            "confidence": "error",
            "detected_buildings": [],
            "reason": f"Gemini 분석 오류: {error_msg}",
            "filter_method": "gemini_error"
        }


def analyze_with_text_only_gemini(place_name: str, address: str = "") -> dict:
    """
    💰 최저비용 옵션: 이미지 없이 텍스트만으로 Gemini 분석
    """
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY가 설정되지 않았습니다.")
    
    # API 키 형식 확인
    if len(GEMINI_API_KEY) < 20:
        raise ValueError(f"GEMINI_API_KEY가 너무 짧습니다. 올바른 API 키인지 확인하세요. (길이: {len(GEMINI_API_KEY)})")
    
    try:
        import google.generativeai as genai
    except ImportError as e:
        error_msg = (
            f"google-generativeai 패키지가 설치되지 않았습니다.\n"
            f"다음 명령어로 설치하세요: pip install google-generativeai"
        )
        raise ImportError(error_msg) from e
    
    # GenerativeModel 실제 사용 가능 여부 확인
    try:
        test_model = genai.GenerativeModel
    except AttributeError:
        # 패키지 버전 확인
        try:
            import pkg_resources
            version = pkg_resources.get_distribution("google-generativeai").version
            error_msg = (
                f"google-generativeai 패키지 버전 ({version})이 Python 3.8과 호환되지 않을 수 있습니다.\n"
                f"다음 명령어로 최신 버전을 설치하세요:\n"
                f"  pip install --upgrade google-generativeai\n"
                f"또는 Python 3.10 이상을 사용하세요."
            )
        except:
            error_msg = (
                f"google-generativeai 패키지가 올바르게 설치되지 않았습니다.\n"
                f"다음 명령어로 재설치하세요:\n"
                f"  pip uninstall google-generativeai\n"
                f"  pip install google-generativeai"
            )
        raise AttributeError(error_msg)
    
    # API 키 설정 (재시도 로직 포함)
    try:
        genai.configure(api_key=GEMINI_API_KEY)
    except Exception as e:
        error_msg = (
            f"Gemini API 키 설정 실패: {str(e)}\n"
            f"API 키 형식 확인:\n"
            f"  - 길이: {len(GEMINI_API_KEY)} 문자\n"
            f"  - 시작 문자: {GEMINI_API_KEY[:5]}...\n"
            f"  - .env 파일에 GEMINI_API_KEY가 올바르게 설정되어 있는지 확인하세요."
        )
        raise ValueError(error_msg) from e
    
    prompt = f"""다음 주차장이 등산객이 실제로 사용할 수 있는 공공 주차장인지 판단해주세요.

장소명: {place_name}
주소: {address}

판단 기준:
- 사용 가능: 공영주차장, 국립공원, 자연휴양림, 관광지 주차장 등
- 사용 불가: 식당, 교회, 사기업, 아파트, 병원 전용 주차장 등

JSON 형식으로만 응답:
{{"is_available": true/false, "confidence": "high/medium/low", "reason": "판단 이유"}}"""

    try:
        # 텍스트 전용 모델 목록 (우선순위 순)
        text_models = [
            "gemini-2.5-flash",
            "gemini-1.5-flash",
            "gemini-1.5-pro",
        ]
        
        # 여러 모델 시도
        last_error = None
        max_retries = 2  # JSON 파싱 실패 시 재시도 횟수
        
        for try_model in text_models:
            for retry in range(max_retries + 1):
                try:
                    model_instance = genai.GenerativeModel(try_model)
                    response = model_instance.generate_content(
                        prompt,
                        generation_config=genai.GenerationConfig(
                            temperature=0.1 + (retry * 0.1),
                            max_output_tokens=200,
                        )
                    )
                    
                    result_text = response.text
                    
                    # 향상된 JSON 추출 함수 사용
                    result = extract_json_from_response(result_text)
                    result["filter_method"] = f"text_only_gemini_{try_model}"
                    if "detected_buildings" not in result:
                        result["detected_buildings"] = []
                    return result
                    
                except json.JSONDecodeError as e:
                    # JSON 파싱 오류 시 재시도
                    if retry < max_retries:
                        time.sleep(0.5)
                        continue
                    last_error = e
                    break  # 다음 모델로
                    
                except Exception as e:
                    last_error = e
                    # NotFound 오류면 다음 모델 시도
                    if "NotFound" in str(e) or "404" in str(e) or "not found" in str(e).lower():
                        break  # 다음 모델로
                    # 다른 오류면 즉시 중단
                    raise
        
        # 모든 모델 실패
        if last_error:
            raise last_error
            
    except ImportError as e:
        error_msg = str(e)
        print(f"\n❌ Gemini 패키지 오류: {error_msg}")
        return {
            "is_available": None,
            "confidence": "error",
            "detected_buildings": [],
            "reason": f"패키지 설치 필요: pip install google-generativeai",
            "filter_method": "text_only_gemini"
        }
    except AttributeError as e:
        error_msg = str(e)
        print(f"\n❌ Gemini 패키지 버전 오류: {error_msg}")
        return {
            "is_available": None,
            "confidence": "error",
            "detected_buildings": [],
            "reason": f"패키지 업그레이드 필요: pip install --upgrade google-generativeai",
            "filter_method": "text_only_gemini"
        }
    except json.JSONDecodeError as e:
        # JSON 파싱 오류 (Gemini 응답이 유효한 JSON이 아닌 경우)
        error_msg = str(e)
        print(f"\n⚠️ JSON 파싱 오류: {error_msg}")
        print(f"   → Gemini 응답이 유효한 JSON 형식이 아닙니다. 재시도하면 해결될 수 있습니다.")
        return {
            "is_available": None,
            "confidence": "error",
            "detected_buildings": [],
            "reason": f"JSON 파싱 오류 (재시도 권장): {error_msg}",
            "filter_method": "text_only_gemini"
        }
    except Exception as e:
        error_type = type(e).__name__
        error_msg = str(e)
        
        # 모델을 찾을 수 없는 경우
        if "NotFound" in error_msg or "404" in error_msg or "not found" in error_msg.lower():
            print(f"\n⚠️ Gemini 모델을 찾을 수 없습니다: {error_msg}")
            print(f"💡 해결 방법:")
            print(f"   1. API 키가 올바른지 확인하세요")
            print(f"   2. Gemini API가 활성화되어 있는지 확인하세요")
        else:
            print(f"\n⚠️ 텍스트 분석 오류 ({error_type}): {error_msg}")
        
        return {
            "is_available": None,
            "confidence": "error",
            "detected_buildings": [],
            "reason": f"텍스트 분석 오류: {error_msg}",
            "filter_method": "text_only_gemini"
        }


# ============================================================
# 지도 이미지 가져오기
# ============================================================

def get_google_static_map_image(lat: float, lng: float, zoom: int = 18, size: str = "640x640", debug: bool = False) -> Optional[bytes]:
    """
    Google Static Maps API를 사용하여 위성/지도 이미지 가져오기
    
    Args:
        lat, lng: 좌표
        zoom: 줌 레벨
        size: 이미지 크기
        debug: 디버그 모드 (에러 상세 출력)
    """
    if not GOOGLE_MAPS_API_KEY:
        if debug:
            print(f"   [Google Maps] API 키가 설정되지 않았습니다.")
        return None
    
    satellite_url = (
        f"https://maps.googleapis.com/maps/api/staticmap"
        f"?center={lat},{lng}"
        f"&zoom={zoom}"
        f"&size={size}"
        f"&maptype=satellite"
        f"&key={GOOGLE_MAPS_API_KEY}"
    )
    
    try:
        response = requests.get(satellite_url, timeout=10)
        if response.status_code == 200:
            if debug:
                print(f"   [Google Maps] ✅ 이미지 가져오기 성공")
            return response.content
        else:
            # 에러 응답 확인
            error_msg = response.text[:200] if response.text else "응답 없음"
            if response.status_code == 403:
                if debug:
                    print(f"   [Google Maps] ❌ API 키 오류 (403): {error_msg}")
                    print(f"      → API 키가 유효하지 않거나 'Maps Static API'가 활성화되지 않았을 수 있습니다.")
            elif response.status_code == 400:
                if debug:
                    print(f"   [Google Maps] ❌ 요청 오류 (400): {error_msg}")
            else:
                if debug:
                    print(f"   [Google Maps] ❌ 오류 ({response.status_code}): {error_msg}")
    except requests.exceptions.Timeout:
        if debug:
            print(f"   [Google Maps] ❌ 타임아웃: 요청 시간 초과")
    except requests.exceptions.ConnectionError:
        if debug:
            print(f"   [Google Maps] ❌ 연결 오류: 인터넷 연결을 확인하세요")
    except Exception as e:
        if debug:
            print(f"   [Google Maps] ❌ 예외 발생: {e}")
    
    return None


def get_kakao_static_map_image(lat: float, lng: float, level: int = 2, width: int = 640, height: int = 640, debug: bool = False) -> Optional[bytes]:
    """
    Kakao Static Maps API를 사용하여 지도 이미지 가져오기
    
    Args:
        lat, lng: 좌표
        level: 줌 레벨
        width, height: 이미지 크기
        debug: 디버그 모드 (에러 상세 출력)
    """
    if not KAKAO_API_KEY:
        if debug:
            print(f"   [Kakao Maps] API 키가 설정되지 않았습니다.")
        return None
    
    url = (
        f"https://dapi.kakao.com/v2/maps/staticmap"
        f"?center={lng},{lat}"  # Kakao는 경도, 위도 순서
        f"&level={level}"
        f"&width={width}"
        f"&height={height}"
        f"&mapType=ROADMAP_HYBRID"
        f"&marker=positions:{lng},{lat}"
    )
    
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            if debug:
                print(f"   [Kakao Maps] ✅ 이미지 가져오기 성공")
            return response.content
        else:
            error_msg = response.text[:200] if response.text else "응답 없음"
            if debug:
                if response.status_code == 401:
                    print(f"   [Kakao Maps] ❌ 인증 오류 (401): API 키가 유효하지 않습니다.")
                elif response.status_code == 403:
                    print(f"   [Kakao Maps] ❌ 권한 오류 (403): {error_msg}")
                else:
                    print(f"   [Kakao Maps] ❌ 오류 ({response.status_code}): {error_msg}")
    except requests.exceptions.Timeout:
        if debug:
            print(f"   [Kakao Maps] ❌ 타임아웃: 요청 시간 초과")
    except requests.exceptions.ConnectionError:
        if debug:
            print(f"   [Kakao Maps] ❌ 연결 오류: 인터넷 연결을 확인하세요")
    except Exception as e:
        if debug:
            print(f"   [Kakao Maps] ❌ 예외 발생: {e}")
    
    return None


# ============================================================
# 메인 검증 함수
# ============================================================

def validate_parking_lots(
    csv_path: str,
    output_path: str = None,
    sample_size: int = None,
    save_images: bool = False,
    image_dir: str = "parking_images",
    use_prefilter: bool = True,
    text_only: bool = False,
):
    """
    CSV 파일의 주차장들을 검증
    
    Args:
        csv_path: 입력 CSV 파일 경로
        output_path: 출력 CSV 파일 경로
        sample_size: 테스트용 샘플 크기
        save_images: 이미지 저장 여부
        image_dir: 이미지 저장 디렉토리
        use_prefilter: 텍스트 사전 필터링 사용
        text_only: 이미지 없이 텍스트만으로 분석
    """
    # CSV 로드
    df = pd.read_csv(csv_path)
    print(f"📊 총 {len(df)}개의 주차장 데이터 로드됨")
    
    # 중복 제거 (같은 place_id는 한 번만 분석)
    unique_places = df.drop_duplicates(subset=['place_id'])[['place_id', 'place_name', 'lat', 'lng', 'address']].copy()
    print(f"🔍 고유 장소: {len(unique_places)}개")
    
    if sample_size:
        unique_places = unique_places.head(sample_size)
        print(f"📌 샘플 크기: {sample_size}개로 제한")
    
    # 설정 표시
    print(f"\n⚙️ 설정:")
    print(f"   - 텍스트 사전 필터링: {'✅ ON' if use_prefilter else '❌ OFF'}")
    print(f"   - 텍스트만 분석: {'✅ ON' if text_only else '❌ OFF'}")
    print(f"   - 🔥 Gemini Flash 사용")
    
    # 패키지 확인
    gemini_available = False
    try:
        import google.generativeai as genai
        # 실제로 사용 가능한지 테스트
        try:
            test_model = genai.GenerativeModel
            gemini_available = True
            # 버전 확인
            try:
                import pkg_resources
                version = pkg_resources.get_distribution("google-generativeai").version
                print(f"   - Gemini 패키지: ✅ 설치됨 (버전 {version})")
            except:
                print(f"   - Gemini 패키지: ✅ 설치됨")
        except AttributeError:
            print(f"\n❌ 오류: google-generativeai 패키지가 올바르게 작동하지 않습니다.")
            try:
                import pkg_resources
                version = pkg_resources.get_distribution("google-generativeai").version
                print(f"   설치된 버전: {version}")
                print(f"   Python 버전: {__import__('sys').version}")
                print(f"\n💡 해결 방법:")
                print(f"   1. Python 3.10 이상 사용 권장")
                print(f"   2. 또는 다음 명령어로 재설치:")
                print(f"      pip uninstall google-generativeai")
                print(f"      pip install google-generativeai")
            except:
                print(f"   다음 명령어로 재설치하세요: pip install --upgrade google-generativeai")
            return None, {}
    except ImportError:
        print(f"\n❌ 오류: google-generativeai 패키지가 설치되지 않았습니다.")
        print(f"   다음 명령어로 설치하세요: pip install google-generativeai Pillow")
        return None, {}
    
    # API 키 상태 확인
    print(f"\n🔑 API 키 상태:")
    if GEMINI_API_KEY:
        key_preview = f"{GEMINI_API_KEY[:8]}...{GEMINI_API_KEY[-4:]}" if len(GEMINI_API_KEY) > 12 else "***"
        print(f"   - Gemini API: ✅ 설정됨 (길이: {len(GEMINI_API_KEY)}, 미리보기: {key_preview})")
        
        # 사용 가능한 모델 확인
        try:
            available_models = get_available_gemini_models()
            if available_models:
                print(f"   - 사용 가능한 모델: {', '.join(available_models[:5])}")
                if len(available_models) > 5:
                    print(f"     ... 외 {len(available_models) - 5}개")
            else:
                print(f"   - ⚠️ 모델 목록 조회 실패 (API 키 확인 필요)")
        except Exception as e:
            print(f"   - ⚠️ 모델 확인 중 오류: {str(e)[:50]}")
    else:
        print(f"   - Gemini API: ❌ 미설정")
        print(f"     → .env 파일에 GEMINI_API_KEY를 설정하세요")
    
    if not text_only:
        print(f"   - Google Maps: {'✅ 설정됨' if GOOGLE_MAPS_API_KEY else '❌ 미설정 (Kakao 사용)'}")
        print(f"   - Kakao Maps: {'✅ 설정됨' if KAKAO_API_KEY else '❌ 미설정'}")
        
        if not GOOGLE_MAPS_API_KEY and not KAKAO_API_KEY:
            print(f"\n⚠️ 경고: 지도 이미지 API 키가 없습니다!")
            print(f"   이미지 분석이 불가능하므로 텍스트만으로 분석합니다.")
            print(f"   .env 파일에 GOOGLE_MAPS_API_KEY 또는 KAKAO_API_KEY를 설정하세요.")
    
    # 이미지 저장 디렉토리 생성
    if save_images:
        Path(image_dir).mkdir(exist_ok=True)
    
    # 결과 저장용
    validation_results = {}
    
    # 통계 카운터
    stats = {
        "prefiltered_available": 0,
        "prefiltered_unavailable": 0,
        "text_only_analyzed": 0,
        "image_analyzed": 0,
        "no_image": 0,
    }
    
    # 이미지 가져오기 실패 경고 (한 번만 출력)
    image_warning_shown = False
    
    # 진행률 표시와 함께 검증
    for idx, row in tqdm(unique_places.iterrows(), total=len(unique_places), desc="주차장 검증 중"):
        place_id = row['place_id']
        place_name = row['place_name']
        lat = row['lat']
        lng = row['lng']
        address = row['address'] if pd.notna(row['address']) else ""
        
        # 💰 1단계: 텍스트 사전 필터링
        if use_prefilter:
            prefilter_result = prefilter_by_text(place_name, address)
            if prefilter_result is not None:
                validation_results[place_id] = prefilter_result
                if prefilter_result["is_available"]:
                    stats["prefiltered_available"] += 1
                else:
                    stats["prefiltered_unavailable"] += 1
                continue
        
        # 💰 2단계: 텍스트만으로 분석
        if text_only:
            result = analyze_with_text_only_gemini(place_name, address)
            validation_results[place_id] = result
            stats["text_only_analyzed"] += 1
            time.sleep(0.1)
            continue
        
        # 3단계: 이미지 기반 분석
        # 첫 번째 시도 시에만 디버그 정보 출력
        debug_first = not image_warning_shown
        image_bytes = get_google_static_map_image(lat, lng, debug=debug_first)
        if image_bytes is None:
            image_bytes = get_kakao_static_map_image(lat, lng, debug=debug_first)
        
        if image_bytes is None:
            # 첫 번째 실패 시에만 경고 출력
            if not image_warning_shown:
                print(f"\n⚠️ 지도 이미지를 가져올 수 없습니다. 텍스트만으로 분석합니다.")
                print(f"\n💡 해결 방법:")
                if not GOOGLE_MAPS_API_KEY and not KAKAO_API_KEY:
                    print(f"   1. .env 파일에 API 키를 설정하세요:")
                    print(f"      GOOGLE_MAPS_API_KEY=your-key-here")
                    print(f"      또는")
                    print(f"      KAKAO_API_KEY=your-key-here")
                else:
                    print(f"   1. API 키가 설정되어 있지만 유효하지 않을 수 있습니다.")
                    print(f"   2. Google Maps: 'Maps Static API'가 활성화되어 있는지 확인하세요.")
                    print(f"   3. Kakao Maps: REST API 키가 올바른지 확인하세요.")
                print(f"\n   (이 경고는 한 번만 표시됩니다)")
                image_warning_shown = True
            
            result = analyze_with_text_only_gemini(place_name, address)
            result["reason"] = f"[이미지 없음] {result.get('reason', '')}"
            validation_results[place_id] = result
            stats["no_image"] += 1
            continue
        
        # 이미지 저장
        if save_images:
            image_path = Path(image_dir) / f"{place_id}.png"
            with open(image_path, 'wb') as f:
                f.write(image_bytes)
        
        # 🔥 Gemini로 이미지 분석
        result = analyze_parking_with_gemini(image_bytes, place_name, address)
        validation_results[place_id] = result
        stats["image_analyzed"] += 1
        
        # 레이트 리밋 (무료: 분당 15회)
        time.sleep(4.5)  # 분당 15회 = 4초 간격
    
    # 결과를 원본 DataFrame에 적용 (깔끔한 한글 칼럼명)
    def get_availability_status(place_id):
        result = validation_results.get(place_id, {})
        is_avail = result.get('is_available')
        if is_avail == True:
            return "사용가능"
        elif is_avail == False:
            return "사용불가능"
        else:
            return "알 수 없음"
    
    df['사용가능여부'] = df['place_id'].map(get_availability_status)
    df['판단이유'] = df['place_id'].map(
        lambda x: validation_results.get(x, {}).get('reason', '')
    )
    
    # 칼럼 순서 재정렬 (원본 칼럼 + 검증 결과를 깔끔하게)
    original_cols = [col for col in df.columns if col not in ['사용가능여부', '판단이유']]
    # 사용가능여부와 판단이유를 url 바로 다음에 배치
    if 'url' in original_cols:
        url_idx = original_cols.index('url')
        new_cols = original_cols[:url_idx+1] + ['사용가능여부', '판단이유'] + original_cols[url_idx+1:]
    else:
        new_cols = original_cols + ['사용가능여부', '판단이유']
    df = df[new_cols]
    
    # 결과 저장
    output_path = output_path or csv_path.replace('.csv', '_validated.csv')
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 검증 완료! 결과 저장됨: {output_path}")
    
    # 통계 출력
    print("\n📈 검증 결과 통계:")
    print(df['사용가능여부'].value_counts())
    
    print("\n💰 분석 방법 통계:")
    print(f"   - 텍스트 필터링 (사용가능): {stats['prefiltered_available']}개 ← 무료!")
    print(f"   - 텍스트 필터링 (사용불가): {stats['prefiltered_unavailable']}개 ← 무료!")
    print(f"   - 텍스트만 Gemini 분석: {stats['text_only_analyzed']}개")
    print(f"   - 이미지 Gemini 분석: {stats['image_analyzed']}개")
    print(f"   - 이미지 없음 (텍스트 대체): {stats['no_image']}개")
    
    total_prefiltered = stats['prefiltered_available'] + stats['prefiltered_unavailable']
    if len(unique_places) > 0:
        savings_pct = (total_prefiltered / len(unique_places)) * 100
        print(f"\n   🎉 텍스트 필터링으로 {savings_pct:.1f}%의 API 호출 절감!")
    
    return df, validation_results


def analyze_single_parking(lat: float, lng: float, place_name: str = "", address: str = ""):
    """단일 주차장 분석 (테스트용)"""
    print(f"🔍 분석 중: {place_name} ({lat}, {lng})")
    
    # API 키 상태 확인
    print(f"\n🔑 API 키 상태:")
    print(f"   - Google Maps: {'✅ 설정됨' if GOOGLE_MAPS_API_KEY else '❌ 미설정'}")
    print(f"   - Kakao Maps: {'✅ 설정됨' if KAKAO_API_KEY else '❌ 미설정'}")
    
    # 이미지 가져오기 (디버그 모드 ON)
    print(f"\n📷 이미지 가져오기 시도:")
    image_bytes = get_google_static_map_image(lat, lng, debug=True)
    if image_bytes is None:
        print(f"   → Google Maps 실패, Kakao Maps 시도 중...")
        image_bytes = get_kakao_static_map_image(lat, lng, debug=True)
    
    if image_bytes is None:
        print(f"\n❌ 이미지를 가져올 수 없습니다.")
        print(f"\n💡 해결 방법:")
        if not GOOGLE_MAPS_API_KEY and not KAKAO_API_KEY:
            print(f"   1. .env 파일에 API 키를 설정하세요:")
            print(f"      GOOGLE_MAPS_API_KEY=your-key-here")
            print(f"      또는")
            print(f"      KAKAO_API_KEY=your-key-here")
        else:
            print(f"   1. API 키가 설정되어 있지만 유효하지 않을 수 있습니다.")
            print(f"   2. Google Maps: 'Maps Static API'가 활성화되어 있는지 확인하세요.")
            print(f"      → https://console.cloud.google.com/apis/library/maps-backend.googleapis.com")
            print(f"   3. Kakao Maps: REST API 키가 올바른지 확인하세요.")
        return None
    
    # 이미지 저장
    with open("test_parking.png", "wb") as f:
        f.write(image_bytes)
    print(f"   ✅ 이미지 저장됨: test_parking.png ({len(image_bytes)} bytes)")
    
    # Gemini 분석
    print(f"\n🤖 Gemini 분석 중...")
    result = analyze_parking_with_gemini(image_bytes, place_name, address)
    print(f"\n📋 분석 결과:")
    print(f"  - 사용 가능: {result.get('is_available')}")
    print(f"  - 확신도: {result.get('confidence')}")
    print(f"  - 발견된 건물: {result.get('detected_buildings')}")
    print(f"  - 판단 이유: {result.get('reason')}")
    
    return result


def estimate_cost(csv_path: str, use_prefilter: bool = True, text_only: bool = False):
    """💰 예상 비용 계산"""
    df = pd.read_csv(csv_path)
    unique_count = df['place_id'].nunique()
    
    print(f"\n💰 예상 비용 계산 (Gemini Flash)")
    print(f"=" * 50)
    print(f"총 고유 주차장: {unique_count}개\n")
    
    # 사전 필터링 예상 비율
    prefilter_rate = 0.6 if use_prefilter else 0
    remaining = int(unique_count * (1 - prefilter_rate))
    
    # Gemini 1.5 Flash 가격
    if text_only:
        cost_per_item = 0.00005  # 텍스트만
        method = "텍스트만 (Gemini Flash)"
    else:
        cost_per_item = 0.0015   # 이미지 포함
        method = "이미지 분석 (Gemini Flash)"
    
    total_cost = remaining * cost_per_item
    
    # 무료 티어 계산
    free_daily = 1500
    free_days = (remaining // free_daily) + 1
    
    print(f"분석 방식: {method}")
    print(f"사전 필터링: {'ON' if use_prefilter else 'OFF'} (예상 {prefilter_rate*100:.0f}% 필터링)")
    print(f"API 호출 예상: {remaining}건")
    print(f"\n🎁 무료 티어: 일 {free_daily}회")
    print(f"   → {free_days}일이면 완전 무료!")
    print(f"\n💵 유료 사용 시: ${total_cost:.2f} (약 ₩{total_cost * 1400:.0f})")
    print(f"=" * 50)

print("✅ 모든 함수 로드 완료!")

✅ 모든 함수 로드 완료!


In [ ]:
# 최저 비용: 텍스트만 분석 (이미지 없음)
estimate_cost(
    csv_path='result/등산로_주차장_검증결과.csv',
    use_prefilter=True,
    text_only=False,
)

In [ ]:
# # 샘플 테스트 (처음 10개만)
# validate_parking_lots(
#     csv_path='result/등산로_주차장.csv',
#     output_path='result/등산로_주차장_샘플10개_검증결과.csv',
#     sample_size=10,
#     save_images=False,
#     use_prefilter=True,
#     text_only=False,
# )

In [ ]:
# 전체 실행 (사전 필터링 + Gemini 이미지 분석)
# 전체 실행 시 대략 8시간 소요
validate_parking_lots(
    csv_path='result/등산로_주차장.csv',
    output_path='result/등산로_주차장_검증결과.csv',
    sample_size=None,  # None = 전체
    save_images=False,
    use_prefilter=True,
    text_only=False,
)

In [ ]:
df_parking_result = pd.read_csv("result/등산로_주차장_검증결과.csv")
df_parking_filtered = df_parking_result[df_parking_result['사용가능여부']=='사용가능'].copy()
df_parking_filtered.to_csv("result/사용가능_주차장.csv")     